In [1]:
# Cell 1: Import required libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split  
from surprise import SVD, Dataset, Reader
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error
import pickle
from typing import Tuple, List, Dict, Set
import torch
import torch.nn as nn
import torch.optim as optim
from collections import defaultdict
from tqdm import tqdm
from sklearn.model_selection import ParameterSampler
import random
from typing import Dict, Tuple, Any
import numpy as np
import pandas as pd
from surprise import SVD, Dataset, Reader
from scipy.sparse import csr_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from typing import Dict, List, Tuple, Any
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares
from sklearn.metrics import mean_squared_error
import numpy as np

In [2]:
recommendations_df = pd.read_pickle('recommendations_processed.pkl')

In [3]:
# Cell 2: Neural Matrix Factorization (NeuMF) Model
class NeuMF(nn.Module):
    def __init__(self, num_users: int, num_items: int, num_factors: int = 32,
                 layers: List[int] = [64, 32, 16, 8], dropout: float = 0.1):
        super(NeuMF, self).__init__()
        
        # GMF part
        self.user_embedding_gmf = nn.Embedding(num_users, num_factors)
        self.item_embedding_gmf = nn.Embedding(num_items, num_factors)
        
        # MLP part
        self.user_embedding_mlp = nn.Embedding(num_users, num_factors)
        self.item_embedding_mlp = nn.Embedding(num_items, num_factors)
        
        # MLP layers
        self.mlp_layers = nn.ModuleList()
        input_size = num_factors * 2
        for layer_size in layers:
            self.mlp_layers.append(nn.Linear(input_size, layer_size))
            self.mlp_layers.append(nn.ReLU())
            self.mlp_layers.append(nn.Dropout(dropout))
            input_size = layer_size
        
        # Prediction layer
        self.prediction = nn.Linear(layers[-1] + num_factors, 1)
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, user_input, item_input):
        # GMF part
        user_embedding_gmf = self.user_embedding_gmf(user_input)
        item_embedding_gmf = self.item_embedding_gmf(item_input)
        gmf_vector = user_embedding_gmf * item_embedding_gmf
        
        # MLP part
        user_embedding_mlp = self.user_embedding_mlp(user_input)
        item_embedding_mlp = self.item_embedding_mlp(item_input)
        mlp_vector = torch.cat([user_embedding_mlp, item_embedding_mlp], dim=-1)
        
        for layer in self.mlp_layers:
            mlp_vector = layer(mlp_vector)
        
        # Concatenate GMF and MLP parts
        vector = torch.cat([gmf_vector, mlp_vector], dim=-1)
        
        # Final prediction
        prediction = self.prediction(vector)
        return prediction.squeeze()
    
    def predict(self, user_idx: int, item_idx: int) -> float:
        self.eval()
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_idx]).to(next(self.parameters()).device)
            item_tensor = torch.LongTensor([item_idx]).to(next(self.parameters()).device)
            prediction = self(user_tensor, item_tensor)
            return prediction.item()

class NeuMFRecommender:
    def __init__(self, num_factors: int = 32, learning_rate: float = 0.001,
                 batch_size: int = 256, epochs: int = 20, device: str = 'cuda' if torch.cuda.is_available() else 'cpu'):
        self.num_factors = num_factors
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.device = device
        self.model = None
        self.optimizer = None
        
    def fit(self, ratings_matrix: np.ndarray, user_mapping: Dict, item_mapping: Dict) -> None:
        num_users, num_items = ratings_matrix.shape
        
        # Create model
        self.model = NeuMF(num_users, num_items, self.num_factors).to(self.device)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        criterion = nn.MSELoss()
        
        # Get non-zero ratings
        user_indices, item_indices = np.nonzero(ratings_matrix)
        ratings = ratings_matrix[user_indices, item_indices]
        
        # Convert to tensors
        user_indices = torch.LongTensor(user_indices).to(self.device)
        item_indices = torch.LongTensor(item_indices).to(self.device)
        ratings = torch.FloatTensor(ratings).to(self.device)
        
        # Training loop
        for epoch in range(self.epochs):
            self.model.train()
            total_loss = 0
            num_batches = len(ratings) // self.batch_size
            
            for i in range(num_batches):
                start_idx = i * self.batch_size
                end_idx = start_idx + self.batch_size
                
                batch_users = user_indices[start_idx:end_idx]
                batch_items = item_indices[start_idx:end_idx]
                batch_ratings = ratings[start_idx:end_idx]
                
                # Forward pass
                predictions = self.model(batch_users, batch_items)
                loss = criterion(predictions, batch_ratings)
                
                # Backward pass
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / num_batches
            if (epoch + 1) % 5 == 0:
                print(f'Epoch {epoch+1}/{self.epochs}, Loss: {avg_loss:.4f}')
    
    def predict(self, user_idx: int, item_idx: int) -> float:
        self.model.eval()
        with torch.no_grad():
            user_tensor = torch.LongTensor([user_idx]).to(self.device)
            item_tensor = torch.LongTensor([item_idx]).to(self.device)
            prediction = self.model(user_tensor, item_tensor)
            return prediction.item()

In [4]:
class ALSRecommender:
    def __init__(self, num_factors=100, regularization=0.01, alpha=40, iterations=15):
        self.num_factors = num_factors
        self.regularization = regularization
        self.alpha = alpha
        self.iterations = iterations
        self.user_factors = None
        self.item_factors = None
        
    def fit(self, ratings_matrix):
        # Convert to sparse matrix if not already
        if not isinstance(ratings_matrix, csr_matrix):
            ratings_matrix = csr_matrix(ratings_matrix)
            
        n_users, n_items = ratings_matrix.shape
        
        # Initialize user and item factors randomly
        self.user_factors = np.random.normal(0, 0.1, (n_users, self.num_factors))
        self.item_factors = np.random.normal(0, 0.1, (n_items, self.num_factors))
        
        # Convert ratings to confidence matrix
        confidence = ratings_matrix.copy()
        confidence.data = 1 + self.alpha * confidence.data
        
        # ALS iterations
        for _ in range(self.iterations):
            # Update user factors
            for u in range(n_users):
                user_items = ratings_matrix[u].indices
                if len(user_items) == 0:
                    continue
                    
                # Get user's item factors and confidence values
                item_factors = self.item_factors[user_items]
                conf = confidence[u, user_items].toarray().flatten()
                
                # Calculate user factors
                A = item_factors.T @ (item_factors * conf[:, np.newaxis])
                A += np.eye(self.num_factors) * self.regularization
                b = (item_factors * conf[:, np.newaxis]).T @ ratings_matrix[u, user_items].toarray().flatten()
                
                self.user_factors[u] = np.linalg.solve(A, b)
            
            # Update item factors
            for i in range(n_items):
                item_users = ratings_matrix[:, i].indices
                if len(item_users) == 0:
                    continue
                    
                # Get item's user factors and confidence values
                user_factors = self.user_factors[item_users]
                conf = confidence[item_users, i].toarray().flatten()
                
                # Calculate item factors
                A = user_factors.T @ (user_factors * conf[:, np.newaxis])
                A += np.eye(self.num_factors) * self.regularization
                b = (user_factors * conf[:, np.newaxis]).T @ ratings_matrix[item_users, i].toarray().flatten()
                
                self.item_factors[i] = np.linalg.solve(A, b)
    
    def predict(self, user_idx, item_idx):
        if self.user_factors is None or self.item_factors is None:
            raise ValueError("Model has not been fitted yet")
        return np.dot(self.user_factors[user_idx], self.item_factors[item_idx])

In [5]:
class ModelBasedRecommender:
    def __init__(self):
        self.svd_model = None
        self.neumf_model = None
        self.als_model = None
        self.user_mapping = None
        self.item_mapping = None
        self.reverse_user_mapping = None
        self.reverse_item_mapping = None
        self.ratings_matrix = None
        self.train_df = None
        self.val_df = None
        self.test_df = None
    
    def setup_data(self, train_df: pd.DataFrame, val_df: pd.DataFrame, test_df: pd.DataFrame) -> None:
        """Set up the data structures needed for the recommender system using pre-split data"""
        self.train_df = train_df
        self.val_df = val_df
        self.test_df = test_df
        
        # Create user and item mappings from training data only
        unique_users = train_df['user_id'].unique()
        unique_items = train_df['app_id'].unique()
        
        self.user_mapping = {user: idx for idx, user in enumerate(unique_users)}
        self.item_mapping = {item: idx for idx, item in enumerate(unique_items)}
        self.reverse_user_mapping = {idx: user for user, idx in self.user_mapping.items()}
        self.reverse_item_mapping = {idx: item for item, idx in self.item_mapping.items()}
        
        # Create ratings matrix from training data
        n_users = len(unique_users)
        n_items = len(unique_items)
        self.ratings_matrix = np.zeros((n_users, n_items))
        
        for _, row in train_df.iterrows():
            if row['user_id'] in self.user_mapping and row['app_id'] in self.item_mapping:
                user_idx = self.user_mapping[row['user_id']]
                item_idx = self.item_mapping[row['app_id']]
                # Combine rating and is_recommended into a single score
                # If is_recommended is 1, boost the rating by 1 point
                # If is_recommended is 0, keep the original rating
                base_rating = row['rating']
                recommendation_boost = row['is_recommended'] * 1.0  # Convert to float
                self.ratings_matrix[user_idx, item_idx] = base_rating + recommendation_boost
    
    def train_svd(self, n_factors: int = 100) -> None:
        """Train SVD model using surprise library."""
        if self.ratings_matrix is None:
            raise ValueError("Must call setup_data before training models")
            
        reader = Reader(rating_scale=(1, 6))  # Updated scale to account for recommendation boost
        ratings_list = []
        for user_idx in range(self.ratings_matrix.shape[0]):
            for item_idx in range(self.ratings_matrix.shape[1]):
                if self.ratings_matrix[user_idx, item_idx] > 0:
                    ratings_list.append({
                        'user': self.reverse_user_mapping[user_idx],
                        'item': self.reverse_item_mapping[item_idx],
                        'rating': self.ratings_matrix[user_idx, item_idx]
                    })
        
        df = pd.DataFrame(ratings_list)
        data = Dataset.load_from_df(df[['user', 'item', 'rating']], reader)
        trainset = data.build_full_trainset()
        
        self.svd_model = SVD(n_factors=n_factors)
        self.svd_model.fit(trainset)
    
    def train_als(self, num_factors=100, alpha=40, regularization=0.01, iterations=15) -> None:
        """Train ALS model."""
        if self.ratings_matrix is None:
            raise ValueError("Must call setup_data before training models")

        self.als_model = ALSRecommender(
            num_factors=num_factors,
            regularization=regularization,
            alpha=alpha,
            iterations=iterations
        )

        # Convert to sparse matrix
        ratings_matrix_sparse = csr_matrix(self.ratings_matrix)
        self.als_model.fit(ratings_matrix_sparse)
    
    def train_neumf(self, n_factors: int = 32, learning_rate: float = 0.001,
                   batch_size: int = 256, epochs: int = 20) -> None:
        """Train NeuMF model."""
        if self.ratings_matrix is None:
            raise ValueError("Must call setup_data before training models")
        
        num_users, num_items = self.ratings_matrix.shape
        
        # Create model
        self.neumf_model = NeuMF(num_users, num_items, n_factors).to('cuda' if torch.cuda.is_available() else 'cpu')
        optimizer = optim.Adam(self.neumf_model.parameters(), lr=learning_rate)
        criterion = nn.MSELoss()
        
        # Get non-zero ratings
        user_indices, item_indices = np.nonzero(self.ratings_matrix)
        ratings = self.ratings_matrix[user_indices, item_indices]
        
        # Convert to tensors
        user_indices = torch.LongTensor(user_indices).to('cuda' if torch.cuda.is_available() else 'cpu')
        item_indices = torch.LongTensor(item_indices).to('cuda' if torch.cuda.is_available() else 'cpu')
        ratings = torch.FloatTensor(ratings).to('cuda' if torch.cuda.is_available() else 'cpu')
        
        # Training loop
        for epoch in range(epochs):
            self.neumf_model.train()
            total_loss = 0
            num_batches = len(ratings) // batch_size
            
            for i in range(num_batches):
                start_idx = i * batch_size
                end_idx = start_idx + batch_size
                
                batch_users = user_indices[start_idx:end_idx]
                batch_items = item_indices[start_idx:end_idx]
                batch_ratings = ratings[start_idx:end_idx]
                
                # Forward pass
                predictions = self.neumf_model(batch_users, batch_items)
                loss = criterion(predictions, batch_ratings)
                
                # Backward pass
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                total_loss += loss.item()
            
            avg_loss = total_loss / num_batches
            if (epoch + 1) % 5 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}')
    
    def get_recommendations(self, user_id: int, n_recommendations: int = 5, 
                          model_type: str = 'svd') -> List[Tuple[int, float]]:
        """Get top N recommendations for a user."""
        if user_id not in self.user_mapping:
            return []
        
        if model_type == 'svd':
            if self.svd_model is None:
                raise ValueError("SVD model not trained")
            predictions = []
            for item_id in self.item_mapping.keys():
                pred = self.svd_model.predict(user_id, item_id).est
                predictions.append((item_id, pred))
        elif model_type == 'als':
            if self.als_model is None:
                raise ValueError("ALS model not trained")
            user_idx = self.user_mapping[user_id]
            predictions = []
            for item_id in self.item_mapping.keys():
                item_idx = self.item_mapping[item_id]
                pred = self.als_model.predict(user_idx, item_idx)
                predictions.append((item_id, pred))
        else:  # neumf
            if self.neumf_model is None:
                raise ValueError("NeuMF model not trained")
            user_idx = self.user_mapping[user_id]
            predictions = []
            for item_idx in range(len(self.item_mapping)):
                pred = self.neumf_model.predict(user_idx, item_idx)
                predictions.append((self.reverse_item_mapping[item_idx], pred))
    
        predictions.sort(key=lambda x: x[1], reverse=True)
        return predictions[:n_recommendations]

In [6]:
def create_cold_start_split(recommendations_df, test_size=0.15, val_size=0.15,
                            timestamp_col='date', cold_start_user_frac=0.01, seed=42):
    import numpy as np
    np.random.seed(seed)

    # Sort by timestamp if available
    if timestamp_col in recommendations_df.columns:
        recommendations_df = recommendations_df.sort_values(timestamp_col)

    # Identify cold-start users and items
    all_users = recommendations_df['user_id'].unique()
    all_items = recommendations_df['app_id'].unique()

    n_cold_users = int(len(all_users) * cold_start_user_frac)

    cold_users = np.random.choice(all_users, size=n_cold_users, replace=False)

    # Cold-start test data
    cold_user_df = recommendations_df[recommendations_df['user_id'].isin(cold_users)]

    # Remove cold users/items from the remaining dataset
    warm_df = recommendations_df[
        ~recommendations_df['user_id'].isin(cold_users)
    ]

    # Now split warm_df by user chronologically
    user_groups = warm_df.groupby('user_id')

    train_data = []
    val_data = []
    test_data = []

    for user_id, user_data in user_groups:
        n = len(user_data)
        if n < 3:
            continue

        n_test = max(1, int(n * test_size))
        n_val = max(1, int(n * val_size))

        user_train = user_data.iloc[:-n_test-n_val]
        user_val = user_data.iloc[-n_test-n_val:-n_test]
        user_test = user_data.iloc[-n_test:]

        train_data.append(user_train)
        val_data.append(user_val)
        test_data.append(user_test)

    train_df = pd.concat(train_data)
    val_df = pd.concat(val_data)
    test_df = pd.concat(test_data)

    return train_df, val_df, test_df, cold_user_df


train_df, val_df, test_df, cold_user_df = create_cold_start_split(recommendations_df)

print(f"Train Set: {len(train_df)} Recommendations")
print(f"Validation Set: {len(val_df)} Recommendations")
print(f"Test Set: {len(test_df)} Recommendations")


Train Set: 77464 Recommendations
Validation Set: 35794 Recommendations
Test Set: 35794 Recommendations


In [7]:
def calculate_metrics_at_k(predictions: Dict[int, List[int]], 
                         ground_truth: Dict[int, Set[int]], 
                         k: int = 5) -> Tuple[float, float, float, float]:
    """
    Calculate Precision, Recall, F1, and NDCG at k for recommendations
    
    Parameters:
    -----------
    predictions : Dict[int, List[int]]
        Dictionary of user_id to list of recommended item_ids
    ground_truth : Dict[int, Set[int]]
        Dictionary of user_id to set of actual item_ids
    k : int
        Number of recommendations to consider
    
    Returns:
    --------
    Tuple[float, float, float, float]
        (precision@k, recall@k, f1@k, ndcg@k)
    """
    precision_list = []
    recall_list = []
    f1_list = []
    ndcg_list = []

    for user in ground_truth:
        if user not in predictions:
            continue
            
        pred = list(predictions[user])[:k]
        truth = ground_truth[user]
        if not truth:
            continue

        # Calculate Precision and Recall
        pred_set = set(pred)
        intersection = pred_set & truth

        precision = len(intersection) / k if len(pred) >= k else len(intersection) / len(pred)
        recall = len(intersection) / len(truth)
        
        # Calculate F1 score
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        # Calculate NDCG
        dcg = 0
        for i, item in enumerate(pred):
            rel = 1 if item in truth else 0
            dcg += rel / np.log2(i + 2)  # i + 2 because i starts from 0
            
        # Calculate IDCG
        ideal_ranking = [1] * min(len(truth), k)
        idcg = sum([rel / np.log2(i + 2) for i, rel in enumerate(ideal_ranking)])
        ndcg = dcg / idcg if idcg > 0 else 0
        
        precision_list.append(precision)
        recall_list.append(recall)
        f1_list.append(f1)
        ndcg_list.append(ndcg)

    if not precision_list:
        return 0.0, 0.0, 0.0, 0.0

    avg_precision = sum(precision_list) / len(precision_list)
    avg_recall = sum(recall_list) / len(recall_list)
    avg_f1 = sum(f1_list) / len(f1_list)
    avg_ndcg = sum(ndcg_list) / len(ndcg_list)

    return avg_precision, avg_recall, avg_f1, avg_ndcg

In [8]:
def evaluate_model(recommender, test_df, model_type, k=5, scenario="Normal"):
    """Helper function to evaluate a model on a specific test set"""
    print(f"\nEvaluating {model_type.upper()} model ({scenario} scenario):")
    predictions = defaultdict(list)
    ground_truth = defaultdict(set)
    
    # Create ground truth
    for _, row in test_df.iterrows():
        if row['is_recommended'] == 1:
            ground_truth[row['user_id']].add(row['app_id'])
    
    # Get predictions
    test_users = test_df['user_id'].unique()
    for user_id in tqdm(test_users, desc=f"Getting {model_type.upper()} predictions"):
        if user_id not in recommender.user_mapping:
            continue
        recs = recommender.get_recommendations(user_id, n_recommendations=k, model_type=model_type)
        predictions[user_id] = [rec[0] for rec in recs]
    
    precision, recall, f1, ndcg = calculate_metrics_at_k(predictions, ground_truth, k=k)
    return {
        'scenario': scenario,
        'model': model_type.upper(),
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'ndcg': ndcg
    }

In [9]:
from sklearn.model_selection import ParameterSampler
import random
from typing import Dict, Tuple, Any

def tune_svd_hyperparameters(recommender, val_df, n_iter=5) -> Tuple[Dict[str, Any], float]:
    """Tune SVD hyperparameters using random search"""
    param_grid = {
        'n_factors': [50, 100, 150, 200],
        'n_epochs': [20, 30, 40, 50],
        'lr_all': [0.001, 0.005, 0.01, 0.05],
        'reg_all': [0.01, 0.02, 0.05, 0.1]
    }
    
    best_score = float('-inf')
    best_params = None
    
    # Generate random parameter combinations
    param_list = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))
    
    for params in param_list:
        # Train model with current parameters
        recommender.train_svd(n_factors=params['n_factors'])
        
        # Evaluate on validation set
        val_results = evaluate_model(recommender, val_df, 'svd', k=5)
        score = val_results['ndcg']  # Using NDCG as the optimization metric
        
        if score > best_score:
            best_score = score
            best_params = params
    
    return best_params, best_score

In [10]:
def tune_als_hyperparameters(recommender, val_df, n_iter=5) -> Tuple[Dict[str, Any], float]:
    """Tune ALS hyperparameters using random search"""
    param_grid = {
        'num_factors': [50, 100, 150, 200],
        'alpha': [10, 20, 40, 60],
        'iterations': [10, 15, 20, 25],
        'regularization': [0.01, 0.05, 0.1, 0.2]
    }
    
    best_score = float('-inf')
    best_params = None
    
    # Generate random parameter combinations
    param_list = list(ParameterSampler(param_grid, n_iter=n_iter, random_state=42))
    
    for params in param_list:
        # Train model with current parameters
        recommender.train_als(
            num_factors=params['num_factors'],
            alpha=params['alpha'],
            iterations=params['iterations'],
            regularization=params['regularization']
        )
        
        # Evaluate on validation set
        val_results = evaluate_model(recommender, val_df, 'als', k=5)
        score = val_results['ndcg']
        
        if score > best_score:
            best_score = score
            best_params = params
    
    return best_params, best_score

In [11]:
# Initialize recommender
recommender = ModelBasedRecommender()

# Load data
print("Loading data...")
with open('recommendations_processed.pkl', 'rb') as f:
    ratings_data = pickle.load(f)

# Create splits using cold start function
print("\nCreating data splits...")
train_df, val_df, test_df, cold_user_df = create_cold_start_split(ratings_data)
print(f"Train set: {len(train_df)} recommendations")
print(f"Validation set: {len(val_df)} recommendations")
print(f"Test set: {len(test_df)} recommendations")

# Setup data
print("\nSetting up data...")
recommender.setup_data(train_df, val_df, test_df)

# Dictionary to store all results
all_results = []

# Train and evaluate each model with hyperparameter tuning
for model_type in ['als', 'svd']:  
    print(f"\nTuning {model_type.upper()} hyperparameters...")
    
    # Perform hyperparameter tuning
    if model_type == 'svd':
        best_params, best_score = tune_svd_hyperparameters(recommender, val_df)
        print(f"Best SVD parameters: {best_params}")
        print(f"Best validation NDCG: {best_score:.4f}")
        recommender.train_svd(n_factors=best_params['n_factors'])
    else:  # als
        best_params, best_score = tune_als_hyperparameters(recommender, val_df)
        print(f"Best ALS parameters: {best_params}")
        print(f"Best validation NDCG: {best_score:.4f}")
        recommender.train_als(
            num_factors=best_params['num_factors'],
            alpha=best_params['alpha'],
            iterations=best_params['iterations'],
            regularization=best_params['regularization']
        )
    
    # Evaluate on all sets with best parameters
    all_results.append(evaluate_model(recommender, test_df, model_type, scenario="Normal"))

# Train NeuMF with default parameters (no tuning)
print("\nTraining NeuMF model with default parameters...")
recommender.train_neumf()
all_results.append(evaluate_model(recommender, test_df, 'neumf', scenario="Normal"))

# Print comparison table
print("\nModel Comparison:")
print("-" * 80)
print(f"{'Model':<10} {'Precision@k':>12} {'Recall@k':>12} {'F1@k':>12} {'NDCG@k':>12}")
print("-" * 80)
for result in all_results:
    print(f"{result['model']:<10} "
          f"{result['precision']:>12.4f} {result['recall']:>12.4f} "
          f"{result['f1']:>12.4f} {result['ndcg']:>12.4f}")
print("-" * 80)

Loading data...

Creating data splits...
Train set: 77464 recommendations
Validation set: 35794 recommendations
Test set: 35794 recommendations

Setting up data...

Tuning ALS hyperparameters...

Evaluating ALS model (Normal scenario):


Getting ALS predictions: 100%|██████████| 34602/34602 [01:25<00:00, 404.96it/s]



Evaluating ALS model (Normal scenario):


Getting ALS predictions: 100%|██████████| 34602/34602 [01:23<00:00, 416.74it/s]



Evaluating ALS model (Normal scenario):


...

Best SVD parameters: {'reg_all': 0.05, 'n_factors': 100, 'n_epochs': 20, 'lr_all': 0.001}
Best validation NDCG: 0.0088

Evaluating SVD model (Normal scenario):


Getting SVD predictions: 100%|██████████| 34602/34602 [03:55<00:00, 146.62it/s]



Training NeuMF model with default parameters...
Epoch 5/20, Loss: 0.0758
Epoch 10/20, Loss: 0.0226
Epoch 15/20, Loss: 0.0117
Epoch 20/20, Loss: 0.0067

Evaluating NEUMF model (Normal scenario):


Getting NEUMF predictions: 100%|██████████| 34602/34602 [4:21:20<00:00,  2.21it/s]  



Model Comparison:
--------------------------------------------------------------------------------
Model       Precision@k     Recall@k         F1@k       NDCG@k
--------------------------------------------------------------------------------
ALS              0.0020       0.0101       0.0034       0.0062
SVD              0.0029       0.0144       0.0048       0.0083
NEUMF            0.0021       0.0104       0.0035       0.0062
--------------------------------------------------------------------------------
